# N-Queens Problem — Genetic Algorithm (GA)

| | |
|---|---|
| **Name** | Ramya Mercy Rajan |
| **Course** | MSc Software Engineering |
| **University** | University of Europe for Applied Sciences |
| **Professor** | Raja Hashim Ali |

---

## Algorithm Overview

The Genetic Algorithm solves the N-Queens problem by gradually improving a population of possible board arrangements over multiple generations. Instead of checking every possible configuration like DFS, GA searches for good solutions using evolutionary techniques inspired by natural selection.

The algorithm mainly uses three operations:

| Step | Operator | Description |
|------|----------|-------------|
| 1 | **Selection** | The best-performing individuals are selected from the population |
| 2 | **Crossover** | Two parent solutions are combined to create new offspring |
| 3 | **Mutation** | Small random changes are introduced to maintain diversity |

In this implementation, each individual is represented as a list of integers where the index represents the row and the value represents the queen’s column position in that row.

Example for N = 4:

```python
[1, 3, 0, 2]
```

This means:
- Row 0 → Column 1
- Row 1 → Column 3
- Row 2 → Column 0
- Row 3 → Column 2

The fitness value is calculated based on the number of queen conflicts. A board with fewer conflicts has better fitness, while a board with zero conflicts is considered a valid solution.

---

## Fitness Function Optimisation

To improve performance, the program uses attack-count arrays for:
- columns
- left diagonals
- right diagonals

This allows conflicts to be calculated in a single pass through the board, reducing the fitness computation time to **O(N)** for each individual.

---

## Parameters Used

| Parameter | Value |
|---|---|
| Population Size | 200 |
| Maximum Generations | 2000 |
| Mutation Rate | 0.10 |

---

## Complexity

The overall time complexity of the Genetic Algorithm is approximately:

```text
O(generations × population size × N)
```

The actual runtime depends on:
- board size
- mutation rate
- population diversity
- number of generations required to reach a valid solution

Compared to exhaustive DFS, the Genetic Algorithm is much more practical for larger N values because it searches for near-optimal solutions instead of exploring every possible configuration.

## 1. Imports

In [87]:
import time
import random
import psutil
import os

## 2. Utility: Memory Measurement

In [89]:
def measure_memory():
    """Return current process RSS memory in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

## 3. Fitness Function

Uses attack-count arrays for an **O(N) forward pass**.

**Key insight:** when we place queen `r` at column `c`, the counts `col[c]`, `diag1[r-c+N-1]`, `diag2[r+c]` already hold the number of previously placed queens that share that column or diagonal.

In [91]:
def fitness(board):
    N         = len(board)
    col       = [0] * N
    diag1     = [0] * (2 * N)
    diag2     = [0] * (2 * N)
    conflicts = 0

    for r, c in enumerate(board):
        conflicts += col[c] + diag1[r - c + N - 1] + diag2[r + c]
        col[c]              += 1
        diag1[r - c + N-1]  += 1
        diag2[r + c]        += 1

    return -conflicts

## 4. Genetic Operators

### 4a. Crossover

In [93]:
def crossover(parent1, parent2):
    cut   = random.randint(0, len(parent1) - 1)
    child = parent1[:cut] + parent2[cut:]
    return child

### 4b. Mutation

In [95]:
def mutate(board, rate=0.1):
    board = board[:]
    if random.random() < rate:
        r        = random.randint(0, len(board) - 1)
        board[r] = random.randint(0, len(board) - 1)
    return board

### 4c. Selection

In [97]:
def select(population, pop_size):
    return sorted(population, key=fitness, reverse=True)[:pop_size // 2]

## 5. Core Genetic Algorithm Solver

In [99]:
def genetic_algorithm(N, pop_size=200, generations=2000, mutation_rate=0.1):
    start      = time.time()
    mem_before = measure_memory()

    # 1. Initialise random population
    population = [
        [random.randint(0, N - 1) for _ in range(N)]
        for _ in range(pop_size)
    ]

    # 2. Generational loop
    for gen in range(generations):

        # 2a. Check for a perfect solution (early exit)
        for individual in population:
            if fitness(individual) == 0:
                return {
                    "solution":    individual,
                    "time_sec":    round(time.time() - start, 4),
                    "memory_MB":   round(measure_memory() - mem_before, 4),
                    "generations": gen,
                }

        # 2b. Selection - keep fittest 50%
        survivors = select(population, pop_size)

        # 2c. Reproduction - breed until population is full
        next_population = survivors[:]

        while len(next_population) < pop_size:
            parent1 = random.choice(survivors)
            parent2 = random.choice(survivors)
            child   = crossover(parent1, parent2)
            child   = mutate(child, mutation_rate)
            next_population.append(child)

        population = next_population

    # 3. Return best individual found (even if not perfect)
    best = max(population, key=fitness)

    return {
        "solution":    best if fitness(best) == 0 else None,
        "time_sec":    round(time.time() - start, 4),
        "memory_MB":   round(measure_memory() - mem_before, 4),
        "generations": generations,
    }

## 6. Pretty Printer

In [101]:
def print_board(board):
    N = len(board)
    print()
    for row in range(N):
        line = ""
        for col in range(N):
            line += " Q " if board[row] == col else " . "
        print(line)
    print()

## 7. Run Experiments

In [112]:
def run_experiments():
    test_sizes = [10, 30, 50, 100, 200, 500]

    print("=" * 60)
    print("  N-Queens - Genetic Algorithm")
    print("=" * 60)
    print(f"  pop_size={200}, generations={2000}, mutation_rate={0.10}")

    for N in test_sizes:
        print(f"\n>>> N = {N}")
        result = genetic_algorithm(
            N,
            pop_size      = 200,
            generations   = 2000,
            mutation_rate = 0.10,
        )

        solved = result["solution"] is not None
        print(f"  Solved           : {solved}")
        print(f"  Generations used : {result['generations']}")
        print(f"  Time             : {result['time_sec']} s")
        print(f"  Memory delta     : {result['memory_MB']} MB")

        if solved:
            print(f"  Solution         : {result['solution']}")
            if N <= 10:
                print_board(result["solution"])
        else:
            print(f"  Result           : No solution found within "
                  f"{result['generations']} generations.")


run_experiments()

  N-Queens - Genetic Algorithm
  pop_size=200, generations=2000, mutation_rate=0.1

>>> N = 10
  Solved           : True
  Generations used : 1628
  Time             : 4.2285 s
  Memory delta     : 0.4609 MB
  Solution         : [6, 2, 5, 7, 9, 0, 8, 4, 1, 3]

 .  .  .  .  .  .  Q  .  .  . 
 .  .  Q  .  .  .  .  .  .  . 
 .  .  .  .  .  Q  .  .  .  . 
 .  .  .  .  .  .  .  Q  .  . 
 .  .  .  .  .  .  .  .  .  Q 
 Q  .  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  .  Q  . 
 .  .  .  .  Q  .  .  .  .  . 
 .  Q  .  .  .  .  .  .  .  . 
 .  .  .  Q  .  .  .  .  .  . 


>>> N = 30
  Solved           : False
  Generations used : 2000
  Time             : 12.9571 s
  Memory delta     : 0.1055 MB
  Result           : No solution found within 2000 generations.

>>> N = 50
  Solved           : False
  Generations used : 2000
  Time             : 21.7849 s
  Memory delta     : -1.2461 MB
  Result           : No solution found within 2000 generations.

>>> N = 100
  Solved           : False
  Gen